In [1]:
import wrds
import pandas as pd

conn = wrds.Connection()

OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "Gochelsea49!"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [11]:
comp_names = conn.raw_sql("""
    SELECT gvkey, conm
    FROM comp.company
""")

comp_names.shape

(54946, 2)

In [20]:
comp_names.to_csv('comp_names.csv', index=False)

In [19]:
comp_tickers = conn.raw_sql("""
    SELECT gvkey, iid, ticker, exchg
    FROM comp.sec
""")

comp_tickers.head()

ProgrammingError: (psycopg2.errors.UndefinedTable) relation "comp.sec" does not exist
LINE 3:     FROM comp.sec
                 ^

[SQL: 
    SELECT gvkey, iid, ticker, exchg
    FROM comp.sec
]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [14]:
gvkey_permno = conn.raw_sql("""
    SELECT gvkey, lpermno as permno, lpermco as permco
    FROM crsp.ccmxpf_linktable
    WHERE linktype IN ('LU', 'LC') -- Active links
""")

crsp_tickers = conn.raw_sql("""
    SELECT permno, ticker
    FROM crsp.stocknames
""")

# Merge the tables to get GVKEY → PERMNO → Ticker
gvkey_to_ticker = gvkey_permno.merge(crsp_tickers, on="permno", how="left")

gvkey_to_ticker.head()

,gvkey,permno,permco,ticker
0,001000,25881.0,23369.0,AE
1,001001,10015.0,6398.0,AMFD
2,001002,10023.0,22159.0,<NA>
3,001003,10031.0,6672.0,ANTQ
4,001003,10031.0,6672.0,ANTQ


In [16]:
gvkey_to_ticker.shape

(85684, 4)

In [18]:
gvkey_to_ticker.to_csv("gvkey_to_ticker.csv")

In [22]:
#Find apple in comp_names
comp_names[comp_names["conm"].str.contains("nvidia", case=False)]

,gvkey,conm
31615,043473,T-REX 2X INVERS NVIDIA DY TG
31616,043474,T-REX 2X LONG NVIDIA DAIL TG
32741,050504,HARVEST NVIDIA HIGH INCM SHR
32743,050506,HARVEST NVIDIA EHD HI INM SH
39473,117768,NVIDIA CORP


In [23]:
gvkey_to_ticker[gvkey_to_ticker["ticker"].str.contains("BSC", case=False)]

,gvkey,permno,permco,ticker
4606,002370,19108.0,671.0,BSCF
4986,002493,19853.0,673.0,BSCO
4987,002493,19853.0,673.0,BSCO
18325,006725,64522.0,670.0,BSCC
18329,006725,64522.0,670.0,BSCC
21731,007828,57251.0,3139.0,NBSC
21732,007828,57251.0,3139.0,NBSC
21733,007828,57251.0,3139.0,NBSC
21734,007828,57251.0,3139.0,NBSC
34243,011818,68304.0,20282.0,BSC


In [24]:
comp_names[comp_names["gvkey"] == "011818"]

,gvkey,conm
10669,011818,BEAR STEARNS COMPANIES INC


In [ ]:
# 🏢 Create Comprehensive GVKEY to Sector/Industry Mapping
# ========================================================

print("🔍 Creating comprehensive GVKEY mapping with sector/industry data...")

# Enhanced query to get GVKEY, sector, industry, and ticker mapping
gvkey_mapping = conn.raw_sql("""
    SELECT
        c.gvkey,
        c.conm AS company_name,
        c.sic,
        c.gsector AS gics_sector,
        c.gind AS gics_industry, 
        c.gsubind AS gics_subindustry,
        c.naicsh AS naics_code,
        l.lpermno AS permno,
        l.lpermco AS permco,
        s.ticker,
        s.exchcd AS exchange_code,
        -- Add sector names for better readability
        CASE 
            WHEN c.gsector = '10' THEN 'Energy'
            WHEN c.gsector = '15' THEN 'Materials'
            WHEN c.gsector = '20' THEN 'Industrials'
            WHEN c.gsector = '25' THEN 'Consumer Discretionary'
            WHEN c.gsector = '30' THEN 'Consumer Staples'
            WHEN c.gsector = '35' THEN 'Health Care'
            WHEN c.gsector = '40' THEN 'Financials'
            WHEN c.gsector = '45' THEN 'Information Technology'
            WHEN c.gsector = '50' THEN 'Communication Services'
            WHEN c.gsector = '55' THEN 'Utilities'
            WHEN c.gsector = '60' THEN 'Real Estate'
            ELSE 'Other'
        END AS sector_name
    FROM comp.company AS c
    LEFT JOIN crsp.ccmxpf_linktable AS l
        ON c.gvkey = l.gvkey
        AND l.linktype IN ('LU', 'LC')
        AND l.usedflag = 1
    LEFT JOIN crsp.stocknames AS s
        ON l.lpermno = s.permno
    WHERE c.gvkey IS NOT NULL
""")

print(f"✅ Retrieved {len(gvkey_mapping)} company mappings")
print(f"📊 Shape: {gvkey_mapping.shape}")
print(f"🏢 Unique companies: {gvkey_mapping['gvkey'].nunique()}")
print(f"🎯 With tickers: {gvkey_mapping['ticker'].notna().sum()}")

# Display sample data
print(f"\n📋 Sample mapping data:")
gvkey_mapping.head()